# 05-04: AgentCore Runtime Online Evaluations

## Overview

This notebook demonstrates **online evaluation** of a deployed AI agent — evaluating a live agent in real-time against actual user-like queries using the **AgentCore Evaluations** API. Unlike offline evaluations that assess pre-collected data, online evaluations run against a deployed agent's live execution traces to measure production-quality performance.

### What is Online Evaluation?

**Online evaluation** means assessing an agent that is actively deployed and serving requests. The evaluation loop is:

1. **Send live queries** to a deployed agent via the `invoke_agent_runtime` API
2. **Capture execution traces** automatically in CloudWatch (spans, tool calls, model responses)
3. **Run evaluators** against those traces using the AgentCore Evaluations API
4. **Score and analyze** results to assess real-world agent behavior

This contrasts with **offline evaluation**, where you evaluate against static datasets or saved conversation logs without a running agent.

### What is AgentCore Evaluations?

**AgentCore Evaluations** is an AWS service that provides the `evaluate()` API for assessing agent performance using built-in evaluators. Key features include:

- **Built-in Evaluators**: Pre-configured evaluators like `Builtin.Faithfulness` and `Builtin.ToolSelectionAccuracy`
- **Session Span Analysis**: Evaluate agent behavior based on execution traces from CloudWatch logs
- **Objective Scoring**: Get numeric scores, labels, and detailed explanations for each evaluation
- **Token Usage Tracking**: Monitor the cost of running evaluations

### Why Online Evaluations?

Online evaluations are essential for production AI agents because they:

1. **Measure real behavior**: Evaluate the agent as it actually runs, including tool latency, search result variability, and model behavior under real conditions
2. **Catch deployment issues**: Detect problems that only appear in production (e.g., network errors, tool failures, permission issues)
3. **Enable continuous monitoring**: Run evaluations on a schedule to track agent quality over time
4. **Validate deployments**: Confirm a newly deployed agent version meets quality thresholds before routing production traffic

## What You'll Learn

In this module, you will learn how to:

1. **Create a Strands Agent**: Build a city search agent using the `strands-agents` framework with web search capabilities

2. **Deploy to AgentCore Runtime**: Use the `bedrock-agentcore-starter-toolkit` to deploy your agent to AWS managed infrastructure

3. **Invoke the Deployed Agent**: Call your agent via the `invoke_agent_runtime` API with session management

4. **Retrieve Session Spans**: Extract execution traces from CloudWatch logs for evaluation

5. **Run AgentCore Evaluations**: Use the `evaluate()` API with built-in evaluators:
   - `Builtin.Faithfulness` - Assess whether responses are grounded in retrieved tool output
   - `Builtin.ToolSelectionAccuracy` - Evaluate whether the agent used appropriate tools

6. **Analyze Evaluation Results**: Interpret scores, labels, and explanations to understand agent performance

7. **Clean Up Resources**: Properly destroy deployed agents to avoid unnecessary AWS charges

### Prerequisites

Before running this notebook, ensure you have:

- AWS credentials configured with appropriate permissions for Bedrock, AgentCore, and CloudWatch
- Access to Amazon Bedrock foundation models (Nova Micro)
- Python 3.9+ environment

### Module Workflow

<img src="images/img05.png" width="400" align="left">
<br clear="all">

## Section 1: Dependencies Installation

First, we'll install all the required packages for this workshop module. This includes:

- **strands-agents** and **strands-agents-tools**: Framework for building AI agents with tool capabilities
- **bedrock-agentcore** and **bedrock-agentcore-starter-toolkit**: AWS packages for AgentCore Runtime deployment and Evaluations API
- **ddgs**: DuckDuckGo search library for web search functionality
- **boto3**: AWS SDK for Python
- **pandas**: Data manipulation library for handling test cases

In [1]:
# Install required packages
# Using -q flag for quiet output to reduce notebook clutter

%pip install -q -r requirements.txt
%pip install -q bedrock-agentcore bedrock-agentcore-starter-toolkit
%pip install -q ddgs
%pip install -q boto3 pandas

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys

# Strands agents framework
from strands import Agent, tool
from strands.models.bedrock import BedrockModel

# AgentCore packages
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from bedrock_agentcore_starter_toolkit import Runtime

# DuckDuckGo search
from ddgs import DDGS

# Web page fetching
import requests
from bs4 import BeautifulSoup

# AWS and data handling
import boto3
import pandas as pd

# Additional standard library imports we'll need
import json
import uuid
import time
from datetime import datetime

## Section 2: City Search Agent Creation

In this section, we'll create a city search agent that can answer questions about city populations and areas. The agent uses:

- **Web Search Tool**: A custom tool using DuckDuckGo to search for current information
- **Get Page Tool**: A tool to fetch and parse web page content for detailed information
- **BedrockModel**: Amazon Nova Micro for optimized latency and cost
- **System Prompt**: Instructions for the agent to act as a tour guide with structured XML output

### Tool Definitions

We'll start by defining the `web_search` and `get_page` tools using the `@tool` decorator from Strands. The `web_search` tool queries DuckDuckGo and returns the top 5 search results, while `get_page` fetches and extracts text content from URLs.

In [3]:
@tool
def web_search(topic: str) -> str:
    """
    Search DuckDuckGo for information about a topic.
    
    Args:
        topic: Search query string
    
    Returns:
        String containing top 5 search results with titles, URLs, and snippets
    """
    try:
        # Use DuckDuckGo search to find information
        results = DDGS(timeout=5).text(topic, max_results=5)
        
        if not results:
            return "No search results found"
        
        # Format results as a readable string
        result_string = ""
        for i, result in enumerate(results):
            title = result.get('title', 'No title')
            url = result.get('href', 'No URL')
            snippet = result.get('body', 'No description')
            result_string += f"Result {i+1}: {title}\nURL: {url}\nSnippet: {snippet}\n\n"
        
        return result_string
        
    except Exception as e:
        return f"Search error: {str(e)}"

@tool
def get_page(url: str) -> str:
    """
    Fetch and return the raw text content from a URL.
    
    Args:
        url: The URL to fetch content from
    
    Returns:
        String containing the text content of the web page
    """
    try:
        response = requests.get(url)
        response.raise_for_status()
        bs = BeautifulSoup(response.text, 'html.parser')
        return bs.text
    except Exception as e:
        return f"Error fetching page: {str(e)}"

# Verify the tools are defined correctly
print(f"Tool name: {web_search.tool_name}")
print(f"Tool description: {web_search.tool_spec.get('description', 'No description')}")

print(f"\nTool name: {get_page.tool_name}")
print(f"Tool description: {get_page.tool_spec.get('description', 'No description')}")

Tool name: web_search
Tool description: Search DuckDuckGo for information about a topic.

Returns:
    String containing top 5 search results with titles, URLs, and snippets

Tool name: get_page
Tool description: Fetch and return the raw text content from a URL.

Returns:
    String containing the text content of the web page


### Model Configuration

Next, we configure the BedrockModel with Amazon Nova Micro. We choose Nova Micro for this workshop because:

- **Optimized Latency**: Nova Micro provides fast response times, ideal for interactive agent workflows
- **Cost Efficiency**: Lower cost per token compared to larger models, suitable for evaluation scenarios with multiple invocations
- **Sufficient Capability**: Handles city search queries effectively while keeping costs manageable

We also configure timeout settings to ensure reliable API calls.

In [4]:
from botocore.config import Config

# Configure timeout settings for Bedrock API calls
# These settings ensure reliable operation during evaluation scenarios:
# - connect_timeout: Maximum time to establish connection (5 seconds)
# - read_timeout: Maximum time to wait for response (60 seconds for agent reasoning)
# - retries: Single retry attempt to handle transient failures
bedrock_config = Config(
    connect_timeout=5,
    read_timeout=60,
    retries={"max_attempts": 1}
)

# Configure BedrockModel with Amazon Nova Micro
# Model choice rationale:
# - us.amazon.nova-micro-v1:0 provides optimized latency and cost for agent workflows
# - Suitable for city search queries that require web search tool usage
# - Cross-region inference prefix (us.) enables automatic region routing
model_id = "us.amazon.nova-micro-v1:0"

city_search_model = BedrockModel(
    model_id=model_id,
    boto_client_config=bedrock_config
)

### Agent Creation with System Prompt

Now we create the City Search Agent by combining:

1. **System Prompt**: Instructions that define the agent's persona and behavior
   - Acts as a helpful tour guide
   - Uses tools to retrieve current data
   - Outputs structured XML tags for programmatic parsing

2. **Tools**: The `web_search` tool for retrieving city information

3. **Model**: The BedrockModel configured with Amazon Nova Micro

#### XML Output Format

The agent is instructed to output responses with specific XML tags:
- `<response>`: Human-friendly response with context and fun facts
- `<pop>`: Population value (numbers only, no commas)
- `<area>`: Area value in square miles (numbers only)

This structured output format enables programmatic parsing of the agent's responses for evaluation and downstream processing.

In [5]:
# Define the system prompt for the City Search Agent
# This prompt establishes:
# 1. Tour guide persona - friendly, informative responses
# 2. Tool usage requirement - always use tools for current data
# 3. XML output format - structured tags for programmatic parsing

system_prompt = '''You are a helpful tour guide. Customers may ask you about the population and size of cities.
You should use tools to retrieve all data, to make sure it is as current as possible.
Please put your human friendly response in 'response' XML tags, and then follow with your data results in 'pop' and 'area' XML tags, for programatic processing.
The values in the 'pop' and 'area' XML tags should only be numbers, no words or commas.
You may also include a fun fact about the city in question.
For example, your response to "what is the population and size of Somewhereland?" might be:
<response>Somewhereland is a great place to get a hot dog. It has a population of 3000 people, and is 100 square miles.</response>
<pop>3000</pop><area>100</area>
'''

# Create the City Search Agent
# Combining the system prompt, tools, and BedrockModel
city_search_agent = Agent(
    system_prompt=system_prompt,
    tools=[web_search, get_page],
    model=city_search_model
)

print("✓ City Search Agent created successfully!")
print("\nAgent Configuration:")
print(f"  Model: {model_id}")
print(f"  Tools: ['web_search', 'get_page']")
print(f"  System prompt length: {len(system_prompt)} characters")
print("\nExpected output format:")
print("  <response>Human-friendly response with fun facts</response>")
print("  <pop>numeric_population</pop>")
print("  <area>numeric_area</area>")

✓ City Search Agent created successfully!

Agent Configuration:
  Model: us.amazon.nova-micro-v1:0
  Tools: ['web_search', 'get_page']
  System prompt length: 743 characters

Expected output format:
  <response>Human-friendly response with fun facts</response>
  <pop>numeric_population</pop>
  <area>numeric_area</area>


### Test Agent Invocation

Before deploying to AgentCore Runtime, let's verify that our agent works correctly by testing it with a sample city query.

This test invocation will:
1. Send a query asking about Seattle, WA's population and area
2. Display the full response from the agent
3. Show the XML tags (`<response>`, `<pop>`, `<area>`) in the output

A successful test confirms that:
- The agent can use the `web_search` tool to retrieve information
- The response includes the expected XML tag structure
- The model is properly configured and accessible

In [6]:
# Test the City Search Agent with a sample query
# This verifies the agent works correctly before deployment to AgentCore Runtime

# Sample query about Seattle, WA
test_query = "What is the population and area of Seattle, WA?"

print("Testing City Search Agent...")
print(f"Query: {test_query}")
print("\n" + "="*60)
print("Agent Response:")
print("="*60 + "\n")

# Invoke the agent with the test query
test_response = city_search_agent(test_query)

# Display the full response
print(test_response)

print("\n" + "="*60)
print("Test Complete!")
print("="*60)

# Verify XML tags are present in the response
response_str = str(test_response)
xml_tags_present = {
    "<response>": "<response>" in response_str,
    "<pop>": "<pop>" in response_str,
    "<area>": "<area>" in response_str
}

print("\nXML Tag Verification:")
for tag, present in xml_tags_present.items():
    status = "✓" if present else "✗"
    print(f"  {status} {tag} tag {'found' if present else 'NOT found'}")

if all(xml_tags_present.values()):
    print("\n✓ Agent test passed! All expected XML tags are present.")
else:
    print("\n⚠ Warning: Some expected XML tags are missing from the response.")

Testing City Search Agent...
Query: What is the population and area of Seattle, WA?

Agent Response:

<thinking> To provide the population and area of Seattle, WA, I need to search for the most recent information on this topic. I will use the web search tool to find the relevant data.</thinking>

Tool #1: web_search
<thinking> The search results provide various resources about Seattle but do not directly include the population and area data. I need to extract the relevant information from an appropriate page. I will try accessing the official city website for the most accurate and up-to-date information.</thinking> 
Tool #2: get_page
<thinking> It appears that the direct link to the population and area data for Seattle, WA resulted in a 404 error, indicating the page might be outdated or moved. I will try searching again with a slightly different query to find the latest population and area data.</thinking> 
Tool #3: web_search
<thinking> The search results provide insights into Seattl

## Section 3: AgentCore Runtime Deployment

Now that we've verified our City Search Agent works correctly, we'll deploy it to **AgentCore Runtime**. This deployment process involves:

1. **Create Agent File**: Write a Python file (`citysearch.py`) that wraps our agent with `BedrockAgentCoreApp` integration
2. **Configure Deployment**: Use the `bedrock-agentcore-starter-toolkit` to set up deployment configuration
3. **Launch to Runtime**: Deploy the containerized agent to AgentCore Runtime
4. **Verify Deployment**: Confirm the agent is ready and test invocation via the API

### Why Deploy to AgentCore Runtime?

Deploying to AgentCore Runtime provides several benefits:

- **Managed Infrastructure**: No need to manage servers, containers, or scaling
- **Built-in Observability**: Automatic logging of all agent interactions to CloudWatch
- **Session Management**: Track conversations across multiple invocations
- **Evaluation Integration**: Session spans can be used directly with AgentCore Evaluations API

### Agent File Structure

The `citysearch.py` file follows the AgentCore Runtime pattern:

```python
from bedrock_agentcore.runtime import BedrockAgentCoreApp

app = BedrockAgentCoreApp()

@app.entrypoint
def invoke(payload):
    # Process the request and return response
    pass

if __name__ == "__main__":
    app.run()
```

This pattern enables AgentCore Runtime to:
- Receive incoming requests via the `invoke` function
- Automatically instrument the agent for observability
- Manage the agent lifecycle in a containerized environment

In [7]:
%%writefile ./citysearch.py
# City Search Agent for AgentCore Runtime Deployment
# This file creates a deployable agent using BedrockAgentCoreApp
#
# The agent:
# - Uses DuckDuckGo web search to find city information
# - Can fetch and parse web pages for detailed information
# - Returns responses with XML tags for programmatic parsing
# - Is configured for AgentCore Runtime with the @app.entrypoint decorator

from botocore.config import Config
from ddgs import DDGS
from strands import Agent, tool
from strands.models import BedrockModel
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from boto3.session import Session
import requests
from bs4 import BeautifulSoup

# Get the current AWS region from the boto session
boto_session = Session()
region = boto_session.region_name

# Configure timeout settings for Bedrock API calls
# - connect_timeout: 5 seconds to establish connection
# - read_timeout: 20 seconds for response (optimized for fast queries)
# - retries: Disabled to fail fast and avoid hanging
quick_config = Config(
    connect_timeout=5,
    read_timeout=20,
    retries={"max_attempts": 0}
)


@tool
def web_search(topic: str) -> str:
    """
    Search DuckDuckGo for information about a topic.
    
    Args:
        topic: Search query string
    
    Returns:
        String containing top 5 search results with titles, URLs, and snippets
    """
    try:
        results = DDGS(timeout=5).text(topic, region=region, max_results=5)
        return results if results else "No results found."
    except Exception as e:
        return f"Search error: {str(e)}"


@tool
def get_page(url: str) -> str:
    """
    Fetch and return the raw text content from a URL.
    
    Args:
        url: The URL to fetch content from
    
    Returns:
        String containing the text content of the web page
    """
    try:
        response = requests.get(url)
        response.raise_for_status()
        bs = BeautifulSoup(response.text, 'html.parser')
        return bs.text
    except Exception as e:
        return f"Error fetching page: {str(e)}"


# Configure BedrockModel with Amazon Nova Micro
# Nova Micro is chosen for optimized latency and cost in agent workflows
chatbot_model_name = "us.amazon.nova-micro-v1:0"
chatbot_model = BedrockModel(
    model_id=chatbot_model_name,
    boto_client_config=quick_config
)

# Define the system prompt for the City Search Agent
# This establishes the tour guide persona and XML output format
system_prompt = '''You are a helpful tour guide. Customers may ask you about the population and size of cities.
You should use tools to retrieve all data, to make sure it is as current as possible.
Please put your human friendly response in 'response' XML tags, and then follow with your data results in 'pop' and 'area' XML tags, for programatic processing.
The values in the 'pop' and 'area' XML tags should only be numbers, no words or commas.
You may also include a fun fact about the city in question.
For example, your response to "what is the population and size of Somewhereland?" might be:
<response>Somewhereland is a great place to get a hot dog. It has a population of 3000 people, and is 100 square miles.</response>
<pop>3000</pop><area>100</area>
'''

# Create the City Search Agent with web_search and get_page tools
chatbot = Agent(
    system_prompt=system_prompt,
    tools=[web_search, get_page],
    model=chatbot_model
)

# Initialize the AgentCore Runtime App
# This enables deployment to AgentCore Runtime with built-in observability
app = BedrockAgentCoreApp()


@app.entrypoint
def invoke(payload):
    """
    AgentCore Runtime entrypoint function.
    
    This function is called by AgentCore Runtime when the agent receives a request.
    It extracts the prompt from the payload, invokes the agent, and returns the response.
    
    Args:
        payload: Dictionary containing 'prompt' key with user query
    
    Returns:
        Agent response string with XML tags for parsing
    """
    user_input = payload.get("prompt", "")
    
    # Invoke the agent with the user's query
    response = chatbot(user_input)
    
    # Extract and return the text content from the response
    return response.message["content"][0]["text"]


if __name__ == "__main__":
    # Run the AgentCore Runtime app
    # This starts the agent server when running locally or in a container
    app.run()

Writing ./citysearch.py


In [8]:
# Verify the citysearch.py file was created successfully
import os

agent_file = "./citysearch.py"

if os.path.exists(agent_file):
    file_size = os.path.getsize(agent_file)
    print(f"✓ citysearch.py created successfully!")
    print(f"  File size: {file_size} bytes")
    print(f"  Location: {os.path.abspath(agent_file)}")

else:
    print(f"✗ Error: {agent_file} was not created")

✓ citysearch.py created successfully!
  File size: 4318 bytes
  Location: /Users/saurabks/aws_git_hub_repos_evals/sample-gen-ai-evaluations-workshop/05-framework-specific-evaluations/05-04-AgentCore-Runtime-Evals/citysearch.py


### Configure AgentCore Runtime Deployment

Now we'll configure the deployment using the `bedrock-agentcore-starter-toolkit`. The `Runtime.configure()` method sets up all the necessary AWS resources and configuration for deploying our agent.

#### Configuration Parameters

| Parameter | Description |
|-----------|-------------|
| `entrypoint` | The Python file containing the agent code with `@app.entrypoint` decorator |
| `auto_create_execution_role` | Automatically create an IAM execution role for the agent |
| `auto_create_ecr` | Automatically create an ECR repository for the container image |
| `requirements_file` | Path to the requirements.txt file with dependencies |
| `region` | AWS region for deployment |
| `agent_name` | Name identifier for the deployed agent |

#### What Happens During Configuration

The `configure()` method:
1. Parses the entrypoint file to identify the `BedrockAgentCoreApp` configuration
2. Creates a `.bedrock_agentcore.yaml` configuration file
3. Generates a `Dockerfile` for containerizing the agent
4. Creates a `.dockerignore` file to exclude unnecessary files
5. Sets up IAM roles and ECR repository references (if auto-create is enabled)

After configuration, we'll be ready to launch the agent to AgentCore Runtime.

In [9]:
# Configure AgentCore Runtime deployment
# This sets up all necessary AWS resources and configuration files

from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session

# Get the current AWS region from the boto session
boto_session = Session()
region = boto_session.region_name

# Define the agent name for deployment
# This name will be used to identify the agent in AgentCore Runtime
agent_name = "citysearch"

# Create a Runtime instance for managing the deployment
agentcore_runtime = Runtime()

# Configure the deployment with all required parameters
print(f"Configuring AgentCore Runtime deployment...")
print(f"  Agent name: {agent_name}")
print(f"  Region: {region}")
print(f"  Entrypoint: citysearch.py")
print(f"  Requirements: requirements.txt")
print()

configure_response = agentcore_runtime.configure(
    entrypoint="citysearch.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name=agent_name
)

# Display the configuration result
print("="*60)
print("Configuration Summary")
print("="*60)
print(f"\n✓ Configuration completed successfully!")
print(f"\nGenerated files:")
print(f"  - Config file: {configure_response.config_path}")
print(f"  - Dockerfile: {configure_response.dockerfile_path}")
print(f"  - Docker ignore: {configure_response.dockerignore_path}")
print(f"\nDeployment settings:")
print(f"  - Runtime: {configure_response.runtime}")
print(f"  - Region: {configure_response.region}")
print(f"  - Account ID: {configure_response.account_id}")
print(f"  - Auto-create ECR: {configure_response.auto_create_ecr}")
print(f"\nThe agent is now configured and ready for launch!")

# Display the full configuration response for reference
print("\n" + "="*60)
print("Full Configuration Response:")
print("="*60)
configure_response

Entrypoint parsed: file=/Users/saurabks/aws_git_hub_repos_evals/sample-gen-ai-evaluations-workshop/05-framework-specific-evaluations/05-04-AgentCore-Runtime-Evals/citysearch.py, bedrock_agentcore_name=citysearch
Memory disabled - agent will be stateless
Configuring BedrockAgentCore agent: citysearch


Configuring AgentCore Runtime deployment...
  Agent name: citysearch
  Region: us-west-2
  Entrypoint: citysearch.py
  Requirements: requirements.txt



💡 No container engine found (Docker/Finch/Podman not installed)

✓ Default deployment uses CodeBuild (no container engine needed), For local builds, install Docker, Finch, or 
Podman

Memory disabled
Network mode: PUBLIC


📄 Generated Dockerfile: 
/Users/saurabks/aws_git_hub_repos_evals/sample-gen-ai-evaluations-workshop/05-framework-specific-evaluations/05-04-
AgentCore-Runtime-Evals/Dockerfile

Generated .dockerignore: /Users/saurabks/aws_git_hub_repos_evals/sample-gen-ai-evaluations-workshop/05-framework-specific-evaluations/05-04-AgentCore-Runtime-Evals/.dockerignore
Setting 'citysearch' as default agent
Bedrock AgentCore configured: /Users/saurabks/aws_git_hub_repos_evals/sample-gen-ai-evaluations-workshop/05-framework-specific-evaluations/05-04-AgentCore-Runtime-Evals/.bedrock_agentcore.yaml


Configuration Summary

✓ Configuration completed successfully!

Generated files:
  - Config file: /Users/saurabks/aws_git_hub_repos_evals/sample-gen-ai-evaluations-workshop/05-framework-specific-evaluations/05-04-AgentCore-Runtime-Evals/.bedrock_agentcore.yaml
  - Dockerfile: /Users/saurabks/aws_git_hub_repos_evals/sample-gen-ai-evaluations-workshop/05-framework-specific-evaluations/05-04-AgentCore-Runtime-Evals/Dockerfile
  - Docker ignore: /Users/saurabks/aws_git_hub_repos_evals/sample-gen-ai-evaluations-workshop/05-framework-specific-evaluations/05-04-AgentCore-Runtime-Evals/.dockerignore

Deployment settings:
  - Runtime: None
  - Region: us-west-2
  - Account ID: 820311036677
  - Auto-create ECR: True

The agent is now configured and ready for launch!

Full Configuration Response:


ConfigureResult(config_path=PosixPath('/Users/saurabks/aws_git_hub_repos_evals/sample-gen-ai-evaluations-workshop/05-framework-specific-evaluations/05-04-AgentCore-Runtime-Evals/.bedrock_agentcore.yaml'), dockerfile_path=PosixPath('/Users/saurabks/aws_git_hub_repos_evals/sample-gen-ai-evaluations-workshop/05-framework-specific-evaluations/05-04-AgentCore-Runtime-Evals/Dockerfile'), dockerignore_path=PosixPath('/Users/saurabks/aws_git_hub_repos_evals/sample-gen-ai-evaluations-workshop/05-framework-specific-evaluations/05-04-AgentCore-Runtime-Evals/.dockerignore'), runtime='None', runtime_type=None, region='us-west-2', account_id='820311036677', execution_role=None, ecr_repository=None, auto_create_ecr=True, s3_path=None, auto_create_s3=False, memory_id=None, network_mode='PUBLIC', network_subnets=None, network_security_groups=None, network_vpc_id=None)

### Launch Agent to AgentCore Runtime

Now that the agent is configured, we'll launch it to AgentCore Runtime. The `runtime.launch()` method performs the following steps:

1. **Build Container**: Publishes the Docker image to ECR using CodeBuild (ARM64 architecture)
2. **Create/Update Agent**: Registers or updates the agent in AgentCore Runtime
3. **Deploy Endpoint**: Creates a runtime endpoint for invoking the agent

#### Launch Parameters

| Parameter | Description |
|-----------|-------------|
| `auto_update_on_conflict` | If `True`, automatically updates the agent if it already exists. If `False`, raises an error on conflict. |

#### What to Expect

The launch process typically takes **2-5 minutes** and includes:
- CodeBuild phases: QUEUED → PROVISIONING → DOWNLOAD_SOURCE → BUILD → POST_BUILD → FINALIZING → COMPLETED
- Agent deployment to AgentCore Runtime
- Endpoint creation and readiness polling

After launch completes, we'll store the **agent ARN** for use in subsequent evaluation steps. The agent ARN is required for:
- Invoking the agent via the `invoke_agent_runtime` API
- Constructing the CloudWatch log group name for span retrieval
- Running AgentCore Evaluations

In [10]:
# Launch the agent to AgentCore Runtime
# This step builds the container, deploys to AgentCore, and creates the endpoint
#
# auto_update_on_conflict=True ensures that if the agent already exists,
# it will be updated with the new configuration rather than raising an error

print("Launching agent to AgentCore Runtime...")
print("This process typically takes 2-5 minutes.")
print("\n" + "="*60)
print("Launch Progress:")
print("="*60 + "\n")

# Launch the agent with auto_update_on_conflict=True
# This will update the agent if it already exists
launch_result = agentcore_runtime.launch(auto_update_on_conflict=True)

print("\n" + "="*60)
print("Launch Complete!")
print("="*60)

🚀 Launching Bedrock AgentCore (cloud mode - RECOMMENDED)...
   • Deploy Python code directly to runtime
   • No Docker required (DEFAULT behavior)
   • Production-ready deployment

💡 Deployment options:
   • runtime.launch()                → Cloud (current)
   • runtime.launch(local=True)      → Local development
Memory disabled - skipping memory creation
Starting CodeBuild ARM64 deployment for agent 'citysearch' to account 820311036677 (us-west-2)
Generated image tag: 20260422-151303-894
Setting up AWS resources (ECR repository, execution roles)...
Getting or creating ECR repository for agent: citysearch


Launching agent to AgentCore Runtime...
This process typically takes 2-5 minutes.

Launch Progress:

Repository doesn't exist, creating new ECR repository: bedrock-agentcore-citysearch


ECR repository available: 820311036677.dkr.ecr.us-west-2.amazonaws.com/bedrock-agentcore-citysearch
Getting or creating execution role for agent: citysearch
Using AWS region: us-west-2, account ID: 820311036677
Role name: AmazonBedrockAgentCoreSDKRuntime-us-west-2-672e22fb7a
Role doesn't exist, creating new execution role: AmazonBedrockAgentCoreSDKRuntime-us-west-2-672e22fb7a
Starting execution role creation process for agent: citysearch
✓ Role creating: AmazonBedrockAgentCoreSDKRuntime-us-west-2-672e22fb7a
Creating IAM role: AmazonBedrockAgentCoreSDKRuntime-us-west-2-672e22fb7a
✓ Role created: arn:aws:iam::820311036677:role/AmazonBedrockAgentCoreSDKRuntime-us-west-2-672e22fb7a
✓ Execution policy attached: BedrockAgentCoreRuntimeExecutionPolicy-citysearch
Role creation complete and ready for use with Bedrock AgentCore
Execution role available: arn:aws:iam::820311036677:role/AmazonBedrockAgentCoreSDKRuntime-us-west-2-672e22fb7a
Preparing CodeBuild project and uploading source...
Getting


Launch Complete!


In [11]:
# Extract and store the agent ARN for later use
# The agent ARN is required for:
# - Invoking the agent via invoke_agent_runtime API
# - Constructing CloudWatch log group names
# - Running AgentCore Evaluations

# Extract the agent ARN from the launch result
agent_arn = launch_result.agent_arn

print("Agent ARN Extracted and Stored")
print("="*60)
print(f"\n✓ Agent ARN: {agent_arn}")

# Store the agent ARN using IPython's %store magic
# This persists the variable across notebook sessions
citysearch_agent_arn = agent_arn
%store citysearch_agent_arn

print(f"\n✓ Agent ARN stored as 'citysearch_agent_arn' for use in subsequent cells")

# Display the full launch result for reference
print("\n" + "="*60)
print("Full Launch Result:")
print("="*60)
print(f"\nAgent ARN: {launch_result.agent_arn}")
print(f"Endpoint ARN: {getattr(launch_result, 'endpoint_arn', 'N/A')}")
print(f"ECR Image: {getattr(launch_result, 'ecr_image', 'N/A')}")

# Display the launch result object
launch_result

Agent ARN Extracted and Stored

✓ Agent ARN: arn:aws:bedrock-agentcore:us-west-2:820311036677:runtime/citysearch-4RFbT7Eomj
Stored 'citysearch_agent_arn' (str)

✓ Agent ARN stored as 'citysearch_agent_arn' for use in subsequent cells

Full Launch Result:

Agent ARN: arn:aws:bedrock-agentcore:us-west-2:820311036677:runtime/citysearch-4RFbT7Eomj
Endpoint ARN: N/A
ECR Image: N/A


LaunchResult(mode='codebuild', tag='bedrock_agentcore-citysearch:None', env_vars=None, port=None, runtime=None, ecr_uri='820311036677.dkr.ecr.us-west-2.amazonaws.com/bedrock-agentcore-citysearch:20260422-151303-894', agent_id='citysearch-4RFbT7Eomj', agent_arn='arn:aws:bedrock-agentcore:us-west-2:820311036677:runtime/citysearch-4RFbT7Eomj', codebuild_id='bedrock-agentcore-citysearch-builder:eb5c7e02-d433-493a-8e74-c78a81fd4e8e', build_output=None)

In [12]:
# NOTE!  If you need to delete the local configuration, and start over from scratch, run this cell.
# This can happen if you deploy an agent then delete it, and the above cell is still trying to update the deleted agent.
# This is needed because in the configuration cell, we create local configuration files which are used by the launch cell.
if False:
    import os

    # Remove the config file so launch creates a new agent instead of updating
    config_files = ['.bedrock_agentcore.yaml', 'Dockerfile', '.dockerignore']
    for f in config_files:
        if os.path.exists(f):
            os.remove(f)
            print(f"Deleted {f}")
        else:
            print(f"{f} not found")

    print("\nNow re-run the configure() and launch() cells")

### Verify Agent Status

After launching the agent, we need to verify that it has reached the **READY** status before we can invoke it. The agent goes through several states during deployment:

| Status | Description |
|--------|-------------|
| `CREATING` | Agent is being created and deployed |
| `UPDATING` | Agent is being updated with new configuration |
| `READY` | Agent is ready to receive invocations |
| `CREATE_FAILED` | Agent creation failed |
| `UPDATE_FAILED` | Agent update failed |
| `DELETE_FAILED` | Agent deletion failed |

This cell polls the agent status every 10 seconds until it reaches a terminal state (READY or a failure state). A timeout of 5 minutes is enforced to prevent indefinite waiting.

**Note**: The agent must be in READY status before proceeding to the invocation and evaluation steps.

In [13]:
# Verify agent status is READY before proceeding
# This cell polls the agent status until it reaches a terminal state
#
# Terminal states:
# - READY: Agent is ready for invocations
# - CREATE_FAILED, UPDATE_FAILED, DELETE_FAILED: Deployment failed
#
# Timeout: 5 minutes (300 seconds) to prevent indefinite waiting

import time

# Configuration for status polling
POLL_INTERVAL_SECONDS = 10  # Time between status checks
TIMEOUT_SECONDS = 300       # Maximum wait time (5 minutes)

# Terminal states that indicate deployment is complete (success or failure)
TERMINAL_STATES = ['READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']

print("Verifying Agent Status")
print("="*60)
print(f"Polling interval: {POLL_INTERVAL_SECONDS} seconds")
print(f"Timeout: {TIMEOUT_SECONDS} seconds ({TIMEOUT_SECONDS // 60} minutes)")
print("\nPolling agent status...\n")

# Track elapsed time for timeout
start_time = time.time()
elapsed_time = 0

# Get initial status
status_response = agentcore_runtime.status()
status = status_response.endpoint['status']
print(f"[{elapsed_time:3d}s] Current status: {status}")

# Poll until terminal state or timeout
while status not in TERMINAL_STATES:
    # Check for timeout
    elapsed_time = int(time.time() - start_time)
    if elapsed_time >= TIMEOUT_SECONDS:
        print(f"\n" + "="*60)
        print("⚠ TIMEOUT: Agent did not reach READY status within 5 minutes")
        print("="*60)
        print(f"\nLast known status: {status}")
        print(f"Elapsed time: {elapsed_time} seconds")
        print("\nPossible actions:")
        print("  1. Wait longer and re-run this cell to check status again")
        print("  2. Check AWS Console for AgentCore Runtime deployment status")
        print("  3. Review CloudWatch logs for any deployment errors")
        break
    
    # Wait before next poll
    time.sleep(POLL_INTERVAL_SECONDS)
    
    # Get updated status
    status_response = agentcore_runtime.status()
    status = status_response.endpoint['status']
    elapsed_time = int(time.time() - start_time)
    print(f"[{elapsed_time:3d}s] Current status: {status}")

# Display final result
print("\n" + "="*60)
print("Status Verification Complete")
print("="*60)

if status == 'READY':
    print(f"\n✓ Agent is READY!")
    print(f"  Total wait time: {elapsed_time} seconds")
    print(f"\nThe agent is now ready to receive invocations.")
elif status in ['CREATE_FAILED', 'UPDATE_FAILED', 'DELETE_FAILED']:
    print(f"\n✗ Deployment FAILED with status: {status}")
    print(f"  Total wait time: {elapsed_time} seconds")
    print(f"\nPlease check CloudWatch logs for error details.")
    print(f"You may need to fix the issue and re-run the launch cell.")
else:
    print(f"\n⚠ Status check ended with status: {status}")
    print(f"  This may indicate a timeout or unexpected state.")

# Display the final status
status

Verifying Agent Status
Polling interval: 10 seconds
Timeout: 300 seconds (5 minutes)

Polling agent status...



Retrieved Bedrock AgentCore status for: citysearch


[  0s] Current status: READY

Status Verification Complete

✓ Agent is READY!
  Total wait time: 0 seconds

The agent is now ready to receive invocations.


'READY'

### Define Online Evaluation Metrics

AgentCore provides 14 built-in evaluators that operate at three different levels of the agent execution hierarchy. Understanding these levels is key to choosing the right metrics for online monitoring.

#### Evaluation Levels: Session, Trace, and Tool

Agent execution data in AgentCore follows a hierarchy:

- **Session** — A logical grouping of related interactions from a single user or workflow. A session may contain one or more traces (e.g., a multi-turn conversation).
- **Trace** — A complete record of a single agent execution or request within a session. Each trace contains one or more spans representing individual operations.
- **Tool call (span)** — A span representing the agent's invocation of an external function, API, or capability, capturing tool name, inputs, execution time, and output.

Built-in evaluators are scoped to one of these levels:

| Level | Evaluator | What It Measures |
|-------|-----------|------------------|
| **Session** | `Builtin.GoalSuccessRate` | Whether all user goals were achieved across the entire conversation |
| **Trace** | `Builtin.Coherence` | Logical consistency and cohesion of the response |
| **Trace** | `Builtin.Conciseness` | Whether the response communicates efficiently without unnecessary elaboration |
| **Trace** | `Builtin.ContextRelevance` | Whether retrieved context contains information needed to answer the query |
| **Trace** | `Builtin.Correctness` | Factual accuracy of the response |
| **Trace** | `Builtin.Faithfulness` | Whether the response is consistent with conversation history and tool outputs |
| **Trace** | `Builtin.Harmfulness` | Whether the response contains harmful or dangerous content |
| **Trace** | `Builtin.Helpfulness` | Whether the response is useful and addresses the user's query |
| **Trace** | `Builtin.InstructionFollowing` | Whether the agent followed its system prompt instructions |
| **Trace** | `Builtin.Refusal` | Whether the agent appropriately refused unsafe or out-of-scope requests |
| **Trace** | `Builtin.ResponseRelevance` | Whether the response is relevant to the user's question |
| **Trace** | `Builtin.Stereotyping` | Whether the response contains stereotyping or bias |
| **Tool** | `Builtin.ToolParameterAccuracy` | Whether the agent passed correct parameters to the tool |
| **Tool** | `Builtin.ToolSelectionAccuracy` | Whether the agent chose the right tool for the task |

#### Choosing Metrics for Online Evaluation

For online evaluation of our city search agent, we select one trace-level and one tool-level evaluator that are most relevant to monitoring live behavior:

- **`Builtin.Faithfulness`** (trace-level) — Critical for online monitoring because live web search results vary with each invocation. This evaluator checks whether the agent's response stays consistent with what its tools actually returned, catching hallucinations that only appear under real conditions.
- **`Builtin.ToolSelectionAccuracy`** (tool-level) — Validates that the agent picks the right tool at each step. In production, unexpected queries or edge cases may cause the agent to misuse tools in ways not seen during development.

You can configure up to 10 evaluators per online evaluation configuration. For this workshop, we use these two to keep evaluation cost and latency manageable.

In [14]:
from bedrock_agentcore_starter_toolkit import Evaluation

# Define the online evaluation metrics (built-in AgentCore evaluators)
ONLINE_EVALUATORS = [
    "Builtin.Faithfulness",
    "Builtin.ToolSelectionAccuracy",
]

# Extract the agent ID from the ARN
agent_id = citysearch_agent_arn.split('/')[-1]

# Configuration name for the online evaluation
ONLINE_EVAL_CONFIG_NAME = "citysearch_online_eval"

# Session idle timeout in minutes (1 min for fast feedback during workshop)
SESSION_TIMEOUT_MINUTES = 1

# Check if an online eval config with this name already exists and delete it
control_client = boto3.client('bedrock-agentcore-control', region_name=region)
existing_configs = control_client.list_online_evaluation_configs()
for cfg in existing_configs.get('onlineEvaluationConfigs', []):
    if cfg['onlineEvaluationConfigName'] == ONLINE_EVAL_CONFIG_NAME:
        config_id_to_delete = cfg['onlineEvaluationConfigId']
        print(f"Deleting existing config: {config_id_to_delete}")
        control_client.delete_online_evaluation_config(
            onlineEvaluationConfigId=config_id_to_delete
        )
        # Wait for deletion to complete
        while True:
            try:
                control_client.get_online_evaluation_config(
                    onlineEvaluationConfigId=config_id_to_delete
                )
                print("  Waiting for deletion...")
                time.sleep(5)
            except Exception:
                break
        print("✓ Deleted.\n")

# Create the config using the starter toolkit (handles IAM role + data source)
eval_client = Evaluation()

config = eval_client.create_online_config(
    config_name=ONLINE_EVAL_CONFIG_NAME,
    agent_id=agent_id,
    sampling_rate=100.0,
    evaluator_list=ONLINE_EVALUATORS,
    config_description="Online evaluation for city search agent",
    auto_create_execution_role=True,
    enable_on_create=True,
)

online_eval_config_id = config['onlineEvaluationConfigId']

# Wait for config to become ACTIVE before updating
# The config starts in CREATING state and cannot be updated until ACTIVE
# Without this wait, participants will hit a ConflictException
control_client = boto3.client('bedrock-agentcore-control', region_name=region)
print("Waiting for evaluation config to become ACTIVE...")
for _i in range(24):
    try:
        _cfg = control_client.get_online_evaluation_config(
            onlineEvaluationConfigId=online_eval_config_id
        )
        _status = _cfg.get('status', 'UNKNOWN')
        if _status == 'ACTIVE':
            print(f"  ✓ Config is ACTIVE ({(_i+1)*5}s)")
            break
        print(f"  Status: {_status} ({(_i+1)*5}s)")
    except Exception:
        pass
    time.sleep(5)

# Step 2: Update the config to set session idle timeout to 1 minute
# The starter toolkit doesn't expose this parameter, so we use boto3 directly
control_client.update_online_evaluation_config(
    onlineEvaluationConfigId=online_eval_config_id,
    rule={
        'samplingConfig': {'samplingPercentage': 100.0},
        'sessionConfig': {'sessionTimeoutMinutes': SESSION_TIMEOUT_MINUTES}
    }
)

online_eval_config_id = config['onlineEvaluationConfigId']

print("Online Evaluation Configuration Created")
print("=" * 60)
print(f"  Config name:  {ONLINE_EVAL_CONFIG_NAME}")
print(f"  Config ID:    {online_eval_config_id}")
print(f"  Agent ID:     {agent_id}")
print(f"  Sampling:     100%")
print(f"  Session timeout: {SESSION_TIMEOUT_MINUTES} min")
print(f"  Status:       {config.get('status', 'N/A')}")
print(f"  Evaluators:")
for evaluator in ONLINE_EVALUATORS:
    print(f"    • {evaluator}")
print(f"\n✓ Online evaluation is now running continuously against live traffic.")


Creating online evaluation config: citysearch_online_eval for agent: citysearch-4RFbT7Eomj
Configuration: sampling_rate=100.0%, evaluators=['Builtin.Faithfulness', 'Builtin.ToolSelectionAccuracy']
Creating online evaluation config: citysearch_online_eval for agent: citysearch-4RFbT7Eomj
Auto-creating execution role for config: citysearch_online_eval
Getting or creating evaluation execution role for config: citysearch_online_eval
Using AWS region: us-west-2, account ID: 820311036677
Role name: AgentCoreEvalsSDK-us-west-2-36f561639a
Role doesn't exist, creating new evaluation execution role: AgentCoreEvalsSDK-us-west-2-36f561639a
Creating IAM role: AgentCoreEvalsSDK-us-west-2-36f561639a
✓ Role created: arn:aws:iam::820311036677:role/AgentCoreEvalsSDK-us-west-2-36f561639a
✓ Execution policy attached: AgentCoreEvaluationPolicy-us-west-2-36f561639a
Waiting for IAM role propagation...
Role creation complete and ready for use with Bedrock AgentCore Evaluation
✓ Execution role ready: arn:aws:i

✅ Online evaluation configuration created!

Waiting for evaluation config to become ACTIVE...
  Status: CREATING (5s)
  Status: CREATING (10s)
  ✓ Config is ACTIVE (15s)
Online Evaluation Configuration Created
  Config name:  citysearch_online_eval
  Config ID:    citysearch_online_eval-dzpxGFCFKI
  Agent ID:     citysearch-4RFbT7Eomj
  Sampling:     100%
  Session timeout: 1 min
  Status:       CREATING
  Evaluators:
    • Builtin.Faithfulness
    • Builtin.ToolSelectionAccuracy

✓ Online evaluation is now running continuously against live traffic.


### Test Invocation via invoke_agent_runtime API

Now that the agent is deployed and in READY status, we'll demonstrate a test invocation using the `invoke_agent_runtime` API. This API allows you to:

- **Invoke the deployed agent**: Send queries to your agent running on AgentCore Runtime
- **Manage sessions**: Use unique session IDs to track conversations and generate trace data
- **Receive responses**: Get the agent's response as a streaming body that can be parsed

#### API Parameters

| Parameter | Description |
|-----------|-------------|
| `agentRuntimeArn` | The ARN of the deployed agent (obtained from launch result) |
| `runtimeSessionId` | A unique session identifier (must be 33+ characters) |
| `payload` | JSON-encoded payload containing the prompt |
| `qualifier` | Optional qualifier (default: "DEFAULT") |

#### Session ID Requirements

The `runtimeSessionId` must be at least 33 characters long. We use a UUID-based format (`eval-session-{uuid}`) which ensures:
- Uniqueness across invocations
- Sufficient length (typically 49 characters)
- Easy identification in CloudWatch logs

This test invocation confirms that the deployment was successful and the agent can respond to queries via the API.

In [15]:
# Test invocation via invoke_agent_runtime API
# This demonstrates successful deployment by invoking the agent through the API
#
# The invoke_agent_runtime API:
# - Sends a query to the deployed agent
# - Returns a streaming response body
# - Generates trace data in CloudWatch for later evaluation

import boto3
import json
import uuid
import textwrap

# Create a boto3 client for bedrock-agentcore
# This client provides access to the invoke_agent_runtime API
agentcore_client = boto3.client('bedrock-agentcore', region_name=region)

# Generate a unique session ID (must be 33+ characters)
# Format: eval-session-{uuid} ensures uniqueness and sufficient length
test_session_id = f"eval-session-{uuid.uuid4()}"

# Prepare the test query payload
# The payload must be JSON-encoded with a 'prompt' key
test_query = "What is the population and area of Chicago, IL?"
payload = json.dumps({"prompt": test_query})

print("Test Invocation via invoke_agent_runtime API")
print("="*60)
print(f"\nAgent ARN: {citysearch_agent_arn}")
print(f"Session ID: {test_session_id}")
print(f"Session ID length: {len(test_session_id)} characters (minimum: 33)")
print(f"\nQuery: {test_query}")
print("\n" + "="*60)
print("Invoking agent...")
print("="*60 + "\n")

# Invoke the deployed agent using the invoke_agent_runtime API
response = agentcore_client.invoke_agent_runtime(
    agentRuntimeArn=citysearch_agent_arn,
    runtimeSessionId=test_session_id,
    payload=payload,
    qualifier="DEFAULT"
)

# Read and parse the response
# The response body is a streaming object that needs to be read
response_body = response['response'].read()
response_data = json.loads(response_body)

# Display the response
print("Agent Response:")
print("-"*60)
if isinstance(response_data, str):
    # Response is a string, wrap it for readability
    wrapped = textwrap.fill(response_data, width=80)
    print(wrapped)
else:
    # Response is a dict/object, pretty print it
    formatted = json.dumps(response_data, indent=2)
    print(formatted)

print("\n" + "="*60)
print("Test Invocation Complete!")
print("="*60)

# Display response metadata
print(f"\nResponse Metadata:")
print(f"  HTTP Status Code: {response['ResponseMetadata']['HTTPStatusCode']}")
print(f"  Request ID: {response['ResponseMetadata']['RequestId']}")
print(f"  Runtime Session ID: {response.get('runtimeSessionId', 'N/A')}")
print(f"  Trace ID: {response.get('traceId', 'N/A')}")

# Verify XML tags are present in the response
response_str = str(response_data)
xml_tags_present = {
    "<response>": "<response>" in response_str,
    "<pop>": "<pop>" in response_str,
    "<area>": "<area>" in response_str
}

print("\nXML Tag Verification:")
for tag, present in xml_tags_present.items():
    status_icon = "✓" if present else "✗"
    print(f"  {status_icon} {tag} tag {'found' if present else 'NOT found'}")

if all(xml_tags_present.values()):
    print("\n✓ Deployment test PASSED! Agent is responding correctly via API.")
else:
    print("\n⚠ Warning: Some expected XML tags are missing from the response.")

print("\nThe agent is successfully deployed and responding to API invocations!")

Test Invocation via invoke_agent_runtime API

Agent ARN: arn:aws:bedrock-agentcore:us-west-2:820311036677:runtime/citysearch-4RFbT7Eomj
Session ID: eval-session-213ac7d8-57c0-4ecc-b5ed-c227eff21399
Session ID length: 49 characters (minimum: 33)

Query: What is the population and area of Chicago, IL?

Invoking agent...

Agent Response:
------------------------------------------------------------
<thinking> I was unable to retrieve the data directly from the U.S. Census
Bureau and Wikipedia due to access restrictions. However, I have found several
other sources that provide information on the population and area of Chicago. I
will extract the relevant data from these sources to provide an accurate
response.</thinking>   <response>Chicago is a dynamic city with a population of
approximately 2.7 million people, as of the latest available data. The city is
currently experiencing a slight annual decline, but the metropolitan area
continues to grow, encompassing a vast region with diverse
com

## Section 4: Live Agent Invocation for Online Trace Generation

Now that our agent is deployed and verified, we enter the **online evaluation loop**. We'll invoke the live agent multiple times to simulate real-time production traffic and capture execution traces for evaluation. This section covers:

1. **Load Test Cases**: Read city data from `city_pop.csv` to simulate diverse user queries
2. **Data Cleaning**: Remove Wikipedia footnotes (like `[c]`, `[d]`) from city names
3. **Generate Session IDs**: Create unique session identifiers to track each live invocation
4. **Execute Live Invocations**: Send queries to the deployed agent via the `invoke_agent_runtime` API
5. **Wait for Trace Propagation**: Allow time for CloudWatch to ingest execution spans

### Simulate Production Traffic

Online evaluation requires invoking the live agent under realistic conditions. Multiple invocations with varied queries allow us to:

- **Capture real execution traces**: Each invocation generates spans showing actual tool calls, model responses, and latencies under production conditions
- **Assess faithfulness under variability**: Live web search results change over time — evaluating across multiple queries reveals how well the agent grounds its responses in whatever data it retrieves
- **Detect intermittent failures**: Production issues like tool timeouts, rate limits, or network errors may only surface across multiple invocations
- **Establish performance baselines**: Aggregate scores across invocations provide a statistical baseline for monitoring agent quality over time

### Test Case Data

The `city_pop.csv` file contains real city data used to generate realistic queries:

| Column | Description | Example |
|--------|-------------|--------|
| `city` | City name (may include Wikipedia footnotes) | "New York[c]" |
| `state` | State abbreviation | "NY" |
| `population` | City population | "8,478,072" |
| `land_area_mi2` | Land area in square miles | "300.5" |

In [ ]:
# Load test cases from city_pop.csv
# This cell loads city data and prepares it for agent invocation
#
# Data cleaning steps:
# 1. Load the CSV file using pandas
# 2. Remove Wikipedia footnotes (e.g., [c], [d], [e]) from city names
# 3. Convert population strings to integers (remove commas)
# 4. Select the first 5 cities as test cases

import pandas as pd
import re

# Load the city population data from CSV
csv_path = "city_pop.csv"
print(f"Loading test cases from {csv_path}...")
print("="*60)

# Read the CSV file
city_df = pd.read_csv(csv_path)

print(f"\nTotal cities in dataset: {len(city_df)}")
print(f"Columns: {list(city_df.columns)}")

# Display raw data before cleaning (first 5 rows)
print("\n" + "="*60)
print("Raw Data (before cleaning):")
print("="*60)
print(city_df.head())

# Clean the data: Remove Wikipedia footnotes from city names
# Footnotes appear as [letter] at the end of city names, e.g., "New York[c]"
# Pattern: \[\w+\] matches [c], [d], [e], etc.
def clean_city_name(city_name):
    """Remove Wikipedia footnotes like [c], [d] from city names."""
    return re.sub(r'\[\w+\]', '', city_name).strip()

# Apply cleaning to the city column
city_df['city_clean'] = city_df['city'].apply(clean_city_name)

# Convert population to integer (remove commas and quotes)
def parse_population(pop_str):
    """Convert population string to integer, handling commas and quotes."""
    if isinstance(pop_str, str):
        # Remove commas and quotes
        clean_str = pop_str.replace(',', '').replace('"', '')
        return int(clean_str)
    return int(pop_str)

city_df['population_int'] = city_df['population'].apply(parse_population)

# Select the first 25 cities as test cases
NUM_TEST_CASES = 25
test_cases_df = city_df.head(NUM_TEST_CASES).copy()

# Create a clean display DataFrame with relevant columns
display_df = test_cases_df[['city_clean', 'state', 'population_int', 'land_area_mi2']].copy()
display_df.columns = ['City', 'State', 'Population', 'Area (sq mi)']

print("\n" + "="*60)
print(f"Test Cases (first {NUM_TEST_CASES} cities, cleaned):")
print("="*60)
print(display_df.to_string(index=False))

# Show which cities had footnotes removed
print("\n" + "="*60)
print("Data Cleaning Summary:")
print("="*60)
for idx, row in test_cases_df.iterrows():
    original = row['city']
    cleaned = row['city_clean']
    if original != cleaned:
        print(f"  ✓ Cleaned: '{original}' → '{cleaned}'")
    else:
        print(f"  - No change: '{original}'")

print(f"\n✓ Loaded {NUM_TEST_CASES} test cases successfully!")
print(f"\nThese cities will be used to invoke the agent and generate traces for evaluation.")

Loading test cases from city_pop.csv...

Total cities in dataset: 346
Columns: ['city', 'state', 'population', 'land_area_mi2']

Raw Data (before cleaning):
          city state population land_area_mi2
0  New York[c]    NY  8,478,072         300.5
1  Los Angeles    CA  3,878,704         469.5
2      Chicago    IL  2,721,308         227.7
3      Houston    TX  2,390,125         640.4
4      Phoenix    AZ  1,673,164           518

Test Cases (first 25 cities, cleaned):
         City State  Population Area (sq mi)
     New York    NY     8478072        300.5
  Los Angeles    CA     3878704        469.5
      Chicago    IL     2721308        227.7
      Houston    TX     2390125        640.4
      Phoenix    AZ     1673164          518
 Philadelphia    PA     1573916        134.4
  San Antonio    TX     1526656        498.8
    San Diego    CA     1404452        325.9
       Dallas    TX     1326087        339.6
 Jacksonville    FL     1009833        747.3
   Fort Worth    TX     1008106 

### Generate Unique Session IDs for Live Traffic

In a production environment, every user interaction with the agent creates a unique session. To simulate realistic live traffic, we generate a unique session ID for each test query — just as a real application would for each end-user request.

Session IDs are critical for online evaluation because they:

- **Isolate execution traces**: Each session's spans (tool calls, model responses, latencies) are tagged separately in CloudWatch, enabling per-request evaluation
- **Simulate concurrent users**: Distinct session IDs mimic multiple users hitting the agent simultaneously, revealing behavior under realistic conditions
- **Enable per-session scoring**: AgentCore Evaluations runs evaluators against individual session traces, so each session ID maps to one evaluation result

#### Session ID Format

AgentCore Runtime requires session IDs to be **at least 33 characters** long. We use `eval-session-{uuid}` to mirror how a production app would generate per-request identifiers:

| Component | Length | Example |
|-----------|--------|--------|
| Prefix | 13 chars | `eval-session-` |
| UUID | 36 chars | `550e8400-e29b-41d4-a716-446655440000` |
| **Total** | **49 chars** | `eval-session-550e8400-e29b-41d4-a716-446655440000` |

In [23]:
# Generate unique session IDs for each test case
# Session IDs are used to:
# 1. Track each invocation in CloudWatch logs
# 2. Retrieve session spans for evaluation
# 3. Correlate invocations with evaluation results
#
# Format: eval-session-{uuid}
# - Prefix 'eval-session-' identifies evaluation invocations
# - UUID ensures uniqueness (36 chars)
# - Total length: 49 characters (exceeds 33 char minimum)

import uuid

# Minimum session ID length required by AgentCore Runtime
MIN_SESSION_ID_LENGTH = 33

# Generate a unique session ID for each test case
# Store in a dictionary mapping city name to session ID for easy retrieval
session_ids = {}

# Also store as a list for ordered access
session_id_list = []

print("Generating Unique Session IDs")
print("="*60)
print(f"\nFormat: eval-session-{{uuid}}")
print(f"Minimum required length: {MIN_SESSION_ID_LENGTH} characters")
print("\n" + "-"*60)

for idx, row in test_cases_df.iterrows():
    city = row['city_clean']
    state = row['state']
    
    # Generate a unique session ID using UUID
    session_id = f"eval-session-{uuid.uuid4()}"
    
    # Store the session ID in both dictionary and list
    city_key = f"{city}, {state}"
    session_ids[city_key] = session_id
    session_id_list.append({
        'city': city,
        'state': state,
        'city_key': city_key,
        'session_id': session_id
    })
    
    # Display the generated session ID
    print(f"\n{idx + 1}. {city_key}")
    print(f"   Session ID: {session_id}")
    print(f"   Length: {len(session_id)} characters ✓")

print("\n" + "="*60)
print("Session ID Generation Summary")
print("="*60)
print(f"\n✓ Generated {len(session_ids)} unique session IDs")
print(f"✓ All session IDs meet minimum length requirement ({MIN_SESSION_ID_LENGTH}+ chars)")
print(f"✓ Session IDs stored in 'session_ids' dict and 'session_id_list' for retrieval")

# Verify all session IDs are unique
unique_ids = set(session_ids.values())
if len(unique_ids) == len(session_ids):
    print(f"✓ All session IDs are unique (no duplicates)")
else:
    print(f"⚠ Warning: Found duplicate session IDs!")

# Verify all session IDs meet minimum length
all_valid_length = all(len(sid) >= MIN_SESSION_ID_LENGTH for sid in session_ids.values())
if all_valid_length:
    print(f"✓ All session IDs are {MIN_SESSION_ID_LENGTH}+ characters")
else:
    print(f"⚠ Warning: Some session IDs are too short!")

# Display the session IDs dictionary for reference
print("\n" + "="*60)
print("Session IDs Dictionary (for later retrieval):")
print("="*60)
for city_key, sid in session_ids.items():
    print(f"  '{city_key}': '{sid}'")

Generating Unique Session IDs

Format: eval-session-{uuid}
Minimum required length: 33 characters

------------------------------------------------------------

1. New York, NY
   Session ID: eval-session-ee723e13-c065-4901-9097-e91072ca7e8f
   Length: 49 characters ✓

2. Los Angeles, CA
   Session ID: eval-session-93008352-dafe-4a98-a888-ff4be1f4d278
   Length: 49 characters ✓

3. Chicago, IL
   Session ID: eval-session-d0a05173-2adb-46cf-b000-57301d607842
   Length: 49 characters ✓

4. Houston, TX
   Session ID: eval-session-e5496897-c4d9-4ae5-8d0d-dc78d25d4925
   Length: 49 characters ✓

5. Phoenix, AZ
   Session ID: eval-session-433e3990-2146-45f3-be78-0ab993f132c1
   Length: 49 characters ✓

6. Philadelphia, PA
   Session ID: eval-session-cd94925c-787b-4b74-94d3-a40826205ed8
   Length: 49 characters ✓

7. San Antonio, TX
   Session ID: eval-session-52bea21b-1ef7-4740-b563-541715097db2
   Length: 49 characters ✓

8. San Diego, CA
   Session ID: eval-session-2f3e9e10-63a8-4eaf-a02a-

### Invoke Agent for Each Test Case

Now we'll invoke the deployed agent for each test case to generate trace data for evaluation. This cell:

1. **Loops through test cases**: Iterates over each city in our test dataset
2. **Generates queries**: Creates a standardized query asking about population and area
3. **Invokes the agent**: Calls `invoke_agent_runtime` API with the unique session ID
4. **Captures responses**: Stores the agent's response for each test case
5. **Displays progress**: Shows real-time progress and results

#### Query Format

Each query follows the format:
```
What is the population and area of {city}, {state}?
```

This standardized format ensures consistent agent behavior and enables meaningful evaluation.


In [24]:
# Invoke the agent for each test case
# This cell loops through all test cases and invokes the live agent

import json
import time
import textwrap

# Track invocation statistics
successful_invocations = 0
failed_invocations = 0
latencies = []

print("Invoking Agent for Each Test Case")
print("="*60)
print(f"\nTotal test cases: {len(session_id_list)}")
print(f"Agent ARN: {citysearch_agent_arn}")
print("\n" + "="*60)

for i, test_case in enumerate(session_id_list):
    city = test_case['city']
    state = test_case['state']
    city_key = test_case['city_key']
    session_id = test_case['session_id']
    query = f"What is the population and area of {city}, {state}?"
    
    print(f"\n[{i + 1}/{len(session_id_list)}] {city_key}")
    print("-"*60)
    print(f"Query: {query}")
    print(f"Session ID: {session_id}")
    print("\nInvoking agent...")
    
    try:
        payload = json.dumps({"prompt": query})
        start_time = time.time()
        
        response = agentcore_client.invoke_agent_runtime(
            agentRuntimeArn=citysearch_agent_arn,
            runtimeSessionId=session_id,
            payload=payload,
            qualifier="DEFAULT"
        )
        
        latency = time.time() - start_time
        # Read and store the response for later correlation with evaluation scores
        response_body = response['response'].read()
        try:
            response_text = json.loads(response_body)
        except (json.JSONDecodeError, TypeError):
            response_text = str(response_body)
        
        successful_invocations += 1
        latencies.append(latency)
        
        # Store response preview for correlation with evaluation scores
        session_id_list[i]['response'] = str(response_text)[:200]
        
        print(f"\n✓ Response received (latency: {latency:.2f}s)")
        print(f"  Preview: {str(response_text)[:100]}...")

         # Wait for 30 seconds intentionally for letting the eval results populate on CW Dashboard
        print("Waiting...")
        time.sleep(30) 
        print("Done!")
        
    except Exception as e:
        failed_invocations += 1
        print(f"\n✗ Invocation failed: {str(e)}")
        print("Continuing with next test case...")

# Display summary
print("\n" + "="*60)
print("Invocation Summary")
print("="*60)
print(f"\nTotal test cases: {len(session_id_list)}")
print(f"Successful invocations: {successful_invocations}")
print(f"Failed invocations: {failed_invocations}")
if latencies:
    print(f"Average latency: {sum(latencies)/len(latencies):.2f} seconds")

if successful_invocations == len(session_id_list):
    print(f"\n✓ All {successful_invocations} invocations completed successfully!")
elif successful_invocations > 0:
    print(f"\n⚠ {successful_invocations} of {len(session_id_list)} invocations succeeded.")
else:
    print(f"\n✗ All invocations failed. Please check the agent deployment and try again.")

Invoking Agent for Each Test Case

Total test cases: 25
Agent ARN: arn:aws:bedrock-agentcore:us-west-2:820311036677:runtime/citysearch-4RFbT7Eomj


[1/25] New York, NY
------------------------------------------------------------
Query: What is the population and area of New York, NY?
Session ID: eval-session-ee723e13-c065-4901-9097-e91072ca7e8f

Invoking agent...

✓ Response received (latency: 5.99s)
  Preview: <thinking> It seems I encountered issues accessing the Census Bureau and other pages directly. Since...
Waiting...
Done!

[2/25] Los Angeles, CA
------------------------------------------------------------
Query: What is the population and area of Los Angeles, CA?
Session ID: eval-session-93008352-dafe-4a98-a888-ff4be1f4d278

Invoking agent...

✗ Invocation failed: An error occurred (RuntimeClientError) when calling the InvokeAgentRuntime operation: Received error (500) from runtime. Please check your CloudWatch logs for more information.
Continuing with next test case...

[3/25

## Section 5: Wait for CloudWatch Log Propagation

After invoking the agent, we need to wait for the execution traces to propagate to CloudWatch Logs before we can retrieve them for evaluation.

#### Why Wait?

CloudWatch Logs operates with **eventual consistency**, meaning:

- **Log Ingestion Delay**: Logs are batched and sent to CloudWatch asynchronously
- **Index Propagation**: After ingestion, logs need time to be indexed and become queryable
- **Cross-Region Latency**: If using cross-region inference, additional latency may occur

#### Recommended Wait Time

A **30-second wait** is typically sufficient for:

| Factor | Typical Delay |
|--------|---------------|
| Log batching and transmission | 5-10 seconds |
| CloudWatch ingestion | 5-10 seconds |
| Index propagation | 5-15 seconds |
| **Total** | **15-35 seconds** |

#### What Happens During the Wait

While we wait, AgentCore Runtime is:

1. **Collecting Spans**: Gathering all execution trace data from the agent invocations
2. **Formatting Logs**: Converting spans to CloudWatch-compatible log format
3. **Transmitting Data**: Sending log batches to CloudWatch Logs service
4. **Indexing**: CloudWatch indexes the logs for efficient querying

After the wait completes, we'll be able to retrieve session spans using the CloudWatch Logs API.

## Section 6: Online Evaluation Results Analysis

In this section, we'll analyze the online evaluation results to understand how our agent performs under live conditions. Unlike offline analysis where you compare against static ground truth, online analysis focuses on:

1. **Faithfulness trends**: Are responses consistently grounded in retrieved tool output, or does the agent hallucinate when search results are sparse?
2. **Tool selection patterns**: Does the agent reliably pick the right tools, or do certain queries cause unnecessary tool calls?
3. **Per-session breakdown**: Identify specific invocations that scored poorly to diagnose production issues

### Viewing Results in the AWS Console

AgentCore provides an **Online Evaluations UI** in the AWS Console where you can monitor evaluation results over time. To access it:

1. Open the [Amazon Bedrock console](https://console.aws.amazon.com/bedrock)
2. Navigate to **AgentCore** > **Evaluations** in the left sidebar

<!-- TODO: Insert screenshot img01.png -->
<img src="images/img01.png" width="800" align="left" style="border: 1px solid #ddd; border-radius: 4px; padding: 4px;">
<br clear="all">

The AgentCore Evaluations dashboard shows all online evaluation configurations in your account. From here you can see the config name, status (ACTIVE/DISABLED), creation date, and the agent it monitors. Select the `citysearch_online_eval` configuration to view its details.

3. Select your agent (`citysearch`) to view evaluation history

<!-- TODO: Insert screenshot img02.png -->
<img src="images/img02.png" width="800" align="left" style="border: 1px solid #ddd; border-radius: 4px; padding: 4px;">
<br clear="all">

The evaluation configuration detail page shows the evaluators attached to this config — in our case, `Builtin.Faithfulness` (trace-level) and `Builtin.ToolSelectionAccuracy` (tool-level). You can also see the data source (CloudWatch log group), sampling rate (100%), and session idle timeout (1 minute). Press **View Results** in the top right to see the evaluation scores.

4. Press **View Results** on the top right side of the panel.

<!-- TODO: Insert screenshot img03.png -->
<img src="images/img03.png" width="800" align="left" style="border: 1px solid #ddd; border-radius: 4px; padding: 4px;">
<br clear="all">

The results view lists all evaluated sessions with their Faithfulness and ToolSelectionAccuracy scores. Each row represents one session (one user interaction). You can see the session ID, timestamp, and per-evaluator scores at a glance. Click on any session to drill into its traces.


4. Use the **Traces** tab to drill into one of the traces

<!-- TODO: Insert screenshot of a single trace img04.png and explain the panel with description of evaluation metric that shows why the particular score has been given to this trace. The ToolSelectionAccuracy is not shown as it is calculate on span level -->
<img src="images/img04.png" width="800" align="left" style="border: 1px solid #ddd; border-radius: 4px; padding: 4px;">
<br clear="all">

The trace detail view shows the **Faithfulness** score with a detailed explanation of why the LLM judge assigned that particular score. It analyzes whether the agent's response is consistent with the tool outputs from the conversation history.

> **Note:** `Builtin.ToolSelectionAccuracy` does not appear in the trace-level view because it is a **tool-level (span-level) evaluator** — it scores each individual tool call span, not the overall trace. To see ToolSelectionAccuracy scores, drill into the individual tool call spans within the trace.

### Fetching Results Programmatically

Online evaluation scores are automatically published as CloudWatch Metrics under the `Bedrock-AgentCore/Evaluations` namespace. Rather than parsing raw log entries, we query these pre-calculated metrics directly using `get_metric_data()`.

Each evaluator publishes its scores as a separate metric, dimensioned by `service.name` and `onlineEvaluationConfigId`. This enables dashboarding, alarming, and programmatic threshold checks.

In [ ]:
import boto3
import time
from datetime import datetime, timedelta

cw_client = boto3.client('cloudwatch', region_name=region)

# Wait for session timeout + evaluation processing
WAIT_SECONDS = SESSION_TIMEOUT_MINUTES * 60
print(f"Waiting {WAIT_SECONDS}s for session timeout ({SESSION_TIMEOUT_MINUTES} min)...\n")
time.sleep(WAIT_SECONDS)

end_time = datetime.utcnow()
start_time = end_time - timedelta(weeks=1)

# Build a metric query for each evaluator
metric_queries = []
for i, evaluator in enumerate(ONLINE_EVALUATORS):
    metric_queries.append({
        'Id': f'eval_{i}',
        'MetricStat': {
            'Metric': {
                'Namespace': 'Bedrock-AgentCore/Evaluations',
                'MetricName': evaluator,
                'Dimensions': [
                    {'Name': 'service.name', 'Value': f'{agent_name}.DEFAULT'},
                ]
            },
            'Period': 300,
            'Stat': 'Average',
        },
    })

response = cw_client.get_metric_data(
    MetricDataQueries=metric_queries,
    StartTime=start_time,
    EndTime=end_time,
)

print("Online Evaluation Scores (from CloudWatch Metrics)")
print("=" * 60)

for i, evaluator in enumerate(ONLINE_EVALUATORS):
    result = response['MetricDataResults'][i]
    values = result.get('Values', [])
    if values:
        avg = sum(values) / len(values)
        print(f"  {evaluator}: avg={avg:.2f}  (datapoints={len(values)})")
    else:
        print(f"  {evaluator}: no datapoints yet")

if not any(r.get('Values') for r in response['MetricDataResults']):
    print("\n  ⚠ No metrics found yet. Results may still be processing.")
    print("    Try re-running this cell after a few more minutes.")


### Per-Session Score Breakdown

The aggregate CloudWatch Metrics above show overall trends, but to understand **which specific queries scored low and why**, we query the raw evaluation results from CloudWatch Logs.

Each log entry in the evaluation results log group contains:

| Field | Path | Description |
|-------|------|-------------|
| Evaluator name | `attributes.gen_ai.evaluation.name` | Which evaluator scored this (e.g., `Builtin.Faithfulness`) |
| Score | `attributes.gen_ai.evaluation.score.value` | Numeric score (0.0 to 1.0) |
| Label | `attributes.gen_ai.evaluation.score.label` | Human-readable label (e.g., "Yes", "No", "Generally Yes") |
| Explanation | `attributes.gen_ai.evaluation.explanation` | Detailed reasoning from the LLM judge |
| Session ID | `attributes.session.id` | Links the score back to a specific agent invocation |

This per-session view helps you:
- **Identify problem queries**: Which city searches produced low Faithfulness scores?
- **Diagnose root causes**: Read the LLM judge's explanation to understand *why* a score was low
- **Correlate with responses**: Match scores against the agent's actual output from the invocation step
- **Prioritize fixes**: Focus on the failure patterns that appear most frequently


In [ ]:
# Per-session score breakdown from CloudWatch Logs
# This queries the raw evaluation log entries for detailed per-session results
#
# Why CloudWatch Logs instead of Metrics?
# - Metrics give aggregates (avg, min, max) but not per-session detail
# - Logs contain the full evaluation payload: score, label, AND explanation
# - The explanation tells you exactly why the LLM judge assigned that score

# Wait for evaluation results to be written to CloudWatch Logs
# The aggregate metrics cell (above) waits for session timeout,
# but evaluation results take additional time to process and land in logs
print('Waiting 90s for evaluation results to propagate to CloudWatch Logs...')
time.sleep(90)

logs_client = boto3.client('logs', region_name=region)
eval_log_group = f"/aws/bedrock-agentcore/evaluations/results/{online_eval_config_id}"

print("Per-Session Evaluation Scores")
print("=" * 60)
print(f"Log group: {eval_log_group}\n")

query = """
fields @timestamp, @message
| sort @timestamp desc
| limit 50
"""

start_query = logs_client.start_query(
    logGroupName=eval_log_group,
    startTime=int((time.time() - 86400) * 1000),
    endTime=int(time.time() * 1000),
    queryString=query,
)

# Poll until query completes
status = 'Running'
while status in ['Running', 'Scheduled']:
    time.sleep(2)
    result = logs_client.get_query_results(queryId=start_query['queryId'])
    status = result['status']

# Parse results using AgentCore evaluation log schema
# Scores are nested under attributes with gen_ai.evaluation.* keys
scores_by_session = {}
for row in result['results']:
    msg = next((f['value'] for f in row if f['field'] == '@message'), None)
    if not msg:
        continue
    try:
        entry = json.loads(msg)
        attrs = entry.get('attributes', {})
        evaluator = attrs.get('gen_ai.evaluation.name', 'Unknown')
        score = attrs.get('gen_ai.evaluation.score.value')
        label = attrs.get('gen_ai.evaluation.score.label', 'N/A')
        session = attrs.get('session.id', 'Unknown')

        if score is not None:
            if session not in scores_by_session:
                scores_by_session[session] = {}
            scores_by_session[session][evaluator] = {
                'score': float(score), 'label': label
            }
    except (json.JSONDecodeError, ValueError):
        continue

# Display per-session results
for session_id, evals in scores_by_session.items():
    short_id = session_id[-12:]
    print(f"\nSession ...{short_id}:")
    for evaluator, data in evals.items():
        icon = "✓" if data['score'] >= 0.5 else "✗"
        print(f"  {icon} {evaluator}: {data['score']:.2f} ({data['label']})")

print(f"\n{'=' * 60}")
print(f"Total sessions evaluated: {len(scores_by_session)}")
if scores_by_session:
    all_scores = [s['score'] for evals in scores_by_session.values() for s in evals.values()]
    print(f"Overall average score: {sum(all_scores)/len(all_scores):.2f}")
else:
    print("⚠ No results yet. Evaluation processing can take 5-10 minutes.")
    print("    Wait 2-3 more minutes and re-run this cell.")


## Section 7: Cleanup

After completing the evaluation, it's important to clean up deployed resources to avoid unnecessary AWS charges. This section covers:

1. **Delete the online evaluation config**: Remove the evaluation configuration so it stops processing traces
2. **Destroy the deployed agent**: Remove the agent from AgentCore Runtime
3. **Verify deletion**: Confirm the agent has been removed
4. **Additional cleanup reminders**: Resources that may need manual cleanup

### Important

Always run the cleanup cells when you're done with the workshop to avoid ongoing charges for:
- AgentCore Runtime compute resources
- Online evaluation processing (LLM judge invocations)
- ECR container storage
- CloudWatch log storage

In [ ]:
# Destroy the deployed agent
# This removes the agent from AgentCore Runtime
#
# WARNING: This action is irreversible. The agent will need to be
# re-deployed if you want to use it again.

# Step 1: Delete the online evaluation config
print("Deleting Online Evaluation Config")
print("="*60)
try:
    control_client.delete_online_evaluation_config(
        onlineEvaluationConfigId=online_eval_config_id
    )
    print(f"\n✓ Deleted evaluation config: {online_eval_config_id}")
except Exception as e:
    print(f"\n⚠ Could not delete evaluation config: {e}")

# Step 2: Destroy the agent
print("\nDestroying Deployed Agent")
print("="*60)
print(f"\nAgent to destroy: {agent_name}")
print(f"Agent ARN: {citysearch_agent_arn}")
print("\n" + "="*60)

# Uncomment the following lines to actually destroy the agent:
# print("Destroying agent...")
# destroy_result = agentcore_runtime.destroy()
# print(f"\n✓ Agent destroyed successfully")
# print(f"\nDestroy result: {destroy_result}")

print("\n⚠ Agent destruction is commented out for safety.")
print("\nTo destroy the agent, uncomment the destroy lines above and re-run this cell.")
print("\nAlternatively, you can destroy from the command line:")
print(f"  agentcore destroy --agent-name {agent_name}")

In [ ]:
# Verify agent deletion
# This cell checks if the agent has been successfully removed

print("Verifying Agent Status")
print("="*60)

try:
    status_response = agentcore_runtime.status()
    status = status_response.endpoint.get('status', 'UNKNOWN')
    print(f"\nAgent status: {status}")
    
    if status == 'READY':
        print("\n⚠ Agent is still deployed.")
        print("  Run the destroy cell above to remove it.")
    elif status in ['DELETING', 'DELETED']:
        print("\n✓ Agent is being deleted or has been deleted.")
except Exception as e:
    if 'ResourceNotFoundException' in str(type(e).__name__) or 'not found' in str(e).lower():
        print("\n✓ Agent not found - deletion confirmed!")
    else:
        print(f"\nStatus check result: {str(e)}")
        print("\nThis may indicate the agent has been deleted.")

### Additional Cleanup Reminders

After destroying the agent, you may want to clean up these additional resources:

| Resource | Location | Action |
|----------|----------|--------|
| Online Eval Config | AWS Console > AgentCore > Evaluations | Delete the online evaluation configuration |
| Eval Results Logs | AWS Console > CloudWatch > Log Groups | Delete log groups starting with `/aws/bedrock-agentcore/evaluations/results/` |
| ECR Repository | AWS Console > ECR | Delete the container image repository |
| CloudWatch Logs | AWS Console > CloudWatch > Log Groups | Delete log groups starting with `/aws/bedrock-agentcore/` |
| X-Ray Traces | AWS Console > CloudWatch > X-Ray > Traces | Delete trace groups and indexing rules associated with the agent |
| IAM Roles | AWS Console > IAM > Roles | Delete auto-created execution roles |
| Local Files | This directory | Delete `citysearch.py`, `.bedrock_agentcore.yaml`, `Dockerfile` |

### Verify in AWS Console

To confirm all resources are cleaned up:

1. **AgentCore Runtime**: Check that no agents are listed
2. **ECR**: Verify the repository is deleted
3. **CloudWatch Logs**: Confirm log groups starting with `/aws/bedrock-agentcore/` are removed
4. **X-Ray Traces**: Verify no trace groups, delivery destinations, or indexing rules remain for the agent under CloudWatch > X-Ray > Settings
5. **IAM**: Check for orphaned roles with `bedrock-agentcore` in the name

## Summary

Congratulations! You've completed the AgentCore Runtime Evaluations workshop module. In this notebook, you learned how to:

1. ✓ **Create a Strands Agent** with web search capabilities and XML output format
2. ✓ **Deploy to AgentCore Runtime** using the starter toolkit
3. ✓ **Invoke the deployed agent** via the `invoke_agent_runtime` API
4. ✓ **Configure online evaluations** with built-in evaluators (Faithfulness, ToolSelectionAccuracy)
5. ✓ **Analyze evaluation results** using CloudWatch Metrics and per-session Logs
6. ✓ **Monitor agent quality** through the AgentCore Evaluations console
7. ✓ **Clean up resources** to avoid unnecessary charges

### Key Takeaways

- **AgentCore Runtime** provides managed infrastructure for deploying AI agents with built-in observability
- **AgentCore Evaluations** offers standardized evaluation through the `evaluate()` API
- **Built-in evaluators** like `Builtin.Faithfulness` and `Builtin.ToolSelectionAccuracy` provide objective assessment
- **Online evaluations** run continuously against live traffic, publishing scores as CloudWatch Metrics for monitoring and alerting

### Next Steps

- Explore additional built-in evaluators in the AgentCore Evaluations documentation
- Create custom evaluators for domain-specific assessment
- Integrate evaluations into your CI/CD pipeline for continuous agent quality monitoring
- Review the other workshop modules for more evaluation techniques